# 11.1.3 Response 정보 OpenAI로 생성하기

유사도 검색 결과를 OpenAI Chat Completions API에 전달하여 사용자에게 보여줄 한국어 답변을 생성하는 예제입니다.

이 노트북의 흐름:

1. OpenAI API 키와 클라이언트 준비
2. 유사도 검색 결과를 프롬프트 형식으로 정리
3. `gpt-4o-mini` 모델로 한국어 응답 생성
4. 샘플 검색 결과로 실행 확인


## 1. 패키지 및 API 키 준비

API 키는 코드에 직접 쓰지 않고 `.env` 파일이나 환경변수의 `OPENAI_API_KEY`에서 읽습니다.

필요한 패키지가 없다면 아래 명령을 터미널에서 실행하세요.

```powershell
.\.venv\Scripts\python.exe -m pip install openai python-dotenv
```

In [9]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 API 키를 넣어주세요.")

client = OpenAI(api_key=api_key)

print("OpenAI client ready")

OpenAI client ready


## 2. 검색 결과를 답변으로 바꾸는 함수

`similarity_results`에는 Chroma DB 등에서 가져온 유사도 검색 결과 문장을 리스트로 넣습니다.

In [10]:
def generate_answer(similarity_results, model="gpt-4o-mini"):
    formatted_results = "\n".join(
        f"Result {idx}: {result}" for idx, result in enumerate(similarity_results, start=1)
    )

    print("Formatted similarity results:\n")
    print(formatted_results)

    prompt = f"""
You are an AI assistant.
Based on the following similarity search results, provide a helpful response to the user in Korean.

Similarity Search Results:
{formatted_results}

Answer in Korean:
- 자연스럽고 친절하게 답변하세요.
- 검색 결과의 핵심 정보를 요약하세요.
- 사용자가 추가 질문을 할 수 있도록 마무리하세요.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=500,
        temperature=1.0,
    )

    return response.choices[0].message.content

## 3. 샘플 검색 결과 준비

아래 리스트는 1.2 Chroma 유사도 검색 결과가 들어왔다고 가정한 예시 데이터입니다.

In [11]:
similarity_results = [
    "A luxurious 4 bedroom villa by the beach with private pool.",
    "A spacious 3 bedroom penthouse with panoramic views of the ocean.",
    "A cozy 1 bedroom house in the suburbs with a large garden.",
]

similarity_results

['A luxurious 4 bedroom villa by the beach with private pool.',
 'A spacious 3 bedroom penthouse with panoramic views of the ocean.',
 'A cozy 1 bedroom house in the suburbs with a large garden.']

## 4. OpenAI 모델로 답변 생성

In [12]:
answer = generate_answer(similarity_results)

print("Generated Answer:\n")
print(answer)

Formatted similarity results:

Result 1: A luxurious 4 bedroom villa by the beach with private pool.
Result 2: A spacious 3 bedroom penthouse with panoramic views of the ocean.
Result 3: A cozy 1 bedroom house in the suburbs with a large garden.
Generated Answer:

안녕하세요! 검색 결과를 바탕으로 몇 가지 추천 숙소 정보를 알려드릴게요.

1. **럭셔리 4베드룸 빌라**: 해변 근처에 위치해 있으며, 개인 수영장이 있는 멋진 숙소입니다.
2. **넓은 3베드룸 펜트하우스**: 바다의 전경을 감상할 수 있는 패노라마 뷰가 매력적인 공간입니다.
3. **아늑한 1베드룸 하우스**: 교외에 자리 잡고 있으며, 큰 정원이 있는 아담하고 편안한 집입니다.

각 숙소는 다양한 특성을 가지고 있으니, 원하시는 스타일이나 필요에 맞는 숙소를 선택하시면 좋을 것 같습니다. 추가 질문이나 더 궁금한 사항이 있으시면 언제든지 말씀해 주세요!


## 5. Chroma 검색 결과와 연결하기

1.2 노트북의 `collection.query(...)` 결과를 그대로 사용한다면, 아래 함수로 `metadatas` 값을 답변 생성용 리스트로 바꿀 수 있습니다.

In [13]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


def chroma_results_to_texts(results, metadata_key="description"):
    metadatas = results.get("metadatas", [[]])[0]
    return [metadata.get(metadata_key, str(metadata)) for metadata in metadatas]


In [14]:
base_dir = Path.cwd()
if base_dir.name != "04_vectordb" and (base_dir / "04_vectordb").exists():
    base_dir = base_dir / "04_vectordb"

persist_dir = base_dir / "chroma_store"
collection_name = "Apt103"

chroma_client = chromadb.PersistentClient(path=str(persist_dir))
collection = chroma_client.get_collection(collection_name)

In [15]:
query = "bright house near a university"
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
query_embedding = embedding_model.encode([query])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=["metadatas", "distances"],
)

print("Query:", query)
print("Chroma results ready:", len(results["metadatas"][0]))

Query: bright house near a university
Chroma results ready: 3


In [17]:
# 사용 예시:
similarity_results = chroma_results_to_texts(results)
answer = generate_answer(similarity_results)
print(answer)

Formatted similarity results:

Result 1: A bright 2 bedroom flat near the university with a spacious living area.
Result 2: A sleek studio apartment in a vibrant neighborhood with a shared gym.
Result 3: A modern 1 bedroom loft in the heart of the city with a rooftop terrace.
안녕하세요! 아래의 검색 결과를 바탕으로 추천드릴 수 있는 숙소 정보입니다:

1. **대학 근처의 밝은 2베드룸 아파트**: 넓은 거실 공간이 있어 가족이나 친구와 함께 지내기 좋습니다.
2. **활기찬 동네의 세련된 스튜디오 아파트**: 공유 체육관이 있어 운동하기 편리합니다.
3. **도심의 현대적인 1베드룸 로프트**: 옥상 테라스가 있어 멋진 경치를 즐길 수 있습니다.

이 중에 특별히 관심 가는 옵션이 있으신가요? 추가 질문이 있으시면 언제든지 말씀해 주세요!


## 6. 정리

벡터 DB 검색 결과는 그대로 사용자에게 보여주기보다, LLM을 이용해 자연어 답변으로 재구성하면 더 읽기 쉽습니다.

이번 예제에서는 검색 결과를 프롬프트에 포함하고, OpenAI 모델이 한국어로 요약 답변을 생성하도록 구성했습니다.